<a href="https://colab.research.google.com/github/UrbanInstitute/nccs-data-bmf/blob/standardization_dev/Address_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install usaddress
!git clone https://github.com/GreenBuildingRegistry/usaddress-scourgify.git
%cd usaddress-scourgify
!pip install .
%cd ..

  Using cached usaddress-0.5.16-py3-none-any.whl.metadata (6.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.1 MB/s eta 0:00:00
Cloning into 'usaddress-scourgify'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 250 (delta 20), reused 18 (delta 18), pack-reused 212 (from 1)
Receiving objects: 100% (250/250), 69.38 KiB | 1.00 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/usaddress-scourgify
Processing /content/usaddress-scourgify
  Preparing metadata (setup.py) ... done
  Created wheel for usaddress-scourgify: filename=usaddress_scourgify-0.6.0-py3-none-any.whl size=27110 sha256=52842841951daa5c0721efcf724ccbbadcb347b59ae2a7c33b43a6207539c6ca
  Stored in directory: /root/.cache/pip/wheels/ba/82/73/fab0321055202c20967d93c0d3b64d4363141fec8118c28f05
Successfully built usaddress-s

In [5]:
import usaddress
import pandas as pd

from scourgify import normalize_address_record
from google.colab import files


/usr/local/lib/python3.12/dist-packages/geocoder/uscensus.py:36: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('^\d+', self.address, re.UNICODE)


In [34]:
# load the csv file with 100 samples
df = pd.read_csv("/content/need1.csv", on_bad_lines='skip', nrows=100)

In [35]:
# Parse addresses using usaddress
def parse_address(addr):
    try:
        # Convert to string and handle potential NaN values
        addr_str = str(addr) if pd.notna(addr) else ""
        if not addr_str:
            return {} # Return empty dict for empty/NaN addresses

        tagged_address, address_type = usaddress.tag(addr_str)
        tagged_address['AddressType'] = address_type
        return tagged_address
    except usaddress.RepeatedLabelError as e:
        # Handle specific usaddress parsing errors
        return {'Error': f"Parsing error (RepeatedLabelError): {e}"}
    except Exception as e:
        # Catch any other unexpected errors during parsing
        return {'Error': f"Parsing error: {e}"}

parsed = df['Sample_Addresses'].apply(parse_address)
parsed_df = pd.DataFrame(parsed.tolist())
result_df = pd.concat([df, parsed_df], axis=1)

# Save the result
result_df.to_csv('parsed_addresses.csv', index=False)

In [29]:
# @title Post-processing helpers (v2) — fix floors & suites in line 2

import re
import pandas as pd

def nz(v):
    return "" if pd.isna(v) or v is None else str(v).strip()

# --- ZIP: keep as-is; normalize 9-digit to 5+4; preserve '-0000' ---
def normalize_zip(z: str) -> str:
    s = nz(z).replace(" ", "").replace("\\-", "-")
    if not s:
        return ""
    if re.fullmatch(r"\d{9}", s):             # 9 contiguous digits → 5+4
        return f"{s[:5]}-{s[5:]}"
    if re.fullmatch(r"\d{5}(-\d{4})?", s):    # valid 5 or 5+4 → keep as-is (incl. -0000)
        return s
    return s                                   # leave nonstandard; QA later if needed

# -------- USPS C2: canonical abbreviations for secondary unit designators --------
# (subset of common types; expand if your data needs more)
SEC_ABBR = {
    "APARTMENT":"APT", "APT":"APT",
    "SUITE":"STE", "STE":"STE",
    "UNIT":"UNIT",
    "ROOM":"RM", "RM":"RM",
    "DEPARTMENT":"DEPT", "DEPT":"DEPT",
    "BUILDING":"BLDG", "BLDG":"BLDG",
    "OFFICE":"OFC", "OFC":"OFC",
    "FLOOR":"FL", "FL":"FL", "FLR":"FL",
    "PENTHOUSE":"PH", "PH":"PH",
    "LOBBY":"LBBY", "LBBY":"LBBY",
    "BASEMENT":"BSMT", "BSMT":"BSMT",
    "TRAILER":"TRLR", "TRLR":"TRLR",
    "SPACE":"SPC", "SPC":"SPC",
    "STOP":"STOP",
    "PIER":"PIER",
    "LOT":"LOT",
    "LOWER":"LOWR", "LOWR":"LOWR",
    "UPPER":"UPPR", "UPPR":"UPPR",
    "FRONT":"FRNT", "FRNT":"FRNT",
    "REAR":"REAR",
    "SIDE":"SIDE",
    "HANGAR":"HNGR", "HNGR":"HNGR",
    "KEY":"KEY",
    "SLIP":"SLIP",
    # CMRA / private mailbox
    "PMB":"PMB",
}

# --- Floor extractor (words & numbers) ---
ORDINAL_UNITS = {
    "FIRST":"1","SECOND":"2","THIRD":"3","FOURTH":"4","FIFTH":"5",
    "SIXTH":"6","SEVENTH":"7","EIGHTH":"8","NINTH":"9","TENTH":"10",
    "ELEVENTH":"11","TWELFTH":"12","THIRTEENTH":"13","FOURTEENTH":"14",
    "FIFTEENTH":"15","SIXTEENTH":"16","SEVENTEENTH":"17","EIGHTEENTH":"18",
    "NINETEENTH":"19"
}
ORDINAL_TENS = {"TWENTIETH":"20","THIRTIETH":"30","FORTIETH":"40","FIFTIETH":"50","SIXTIETH":"60","SEVENTIETH":"70","EIGHTIETH":"80","NINETIETH":"90"}
CARDINAL_TENS = {"TWENTY":"20","THIRTY":"30","FORTY":"40","FIFTY":"50","SIXTY":"60","SEVENTY":"70","EIGHTY":"80","NINETY":"90"}

def _ord_words_to_num(words: str):
    w = re.sub(r"[-]+", " ", nz(words).upper())
    if w in ORDINAL_UNITS: return ORDINAL_UNITS[w]
    if w in ORDINAL_TENS:  return ORDINAL_TENS[w]
    parts = w.split()
    if len(parts) == 2 and parts[0] in CARDINAL_TENS and parts[1] in ORDINAL_UNITS:
        return str(int(CARDINAL_TENS[parts[0]]) + int(ORDINAL_UNITS[parts[1]]))
    return None

# Floor patterns in free text
FLOOR_PATTERNS = [
    r"(?i)\b(?:FL(?:OOR)?|FLR)\b[ .#-]*(?P<num1>\d{1,3}[A-Z]?)\b",
    r"(?i)\b(?P<num2>\d{1,3})(?:ST|ND|RD|TH)?\s*(?:FLOOR|FLR)\b",
    r"(?i)\b(?P<word1>[A-Z]+(?:[-\s][A-Z]+)*)\s+FLOOR\b",
    r"(?i)\b(?:FLOOR|FLR)\s+(?P<word2>[A-Z]+(?:[-\s][A-Z]+)*)\b",
]

def extract_floor_text(line1: str):
    """Find any FLOOR phrase inside line 1; return (clean_line1, 'FL n' or None)."""
    s = " " + (line1 or "") + " "
    for pat in FLOOR_PATTERNS:
        m = re.search(pat, s)
        if not m:
            continue
        fl_val = m.groupdict().get("num1") or m.groupdict().get("num2")
        if not fl_val:
            w = m.groupdict().get("word1") or m.groupdict().get("word2")
            fl_val = _ord_words_to_num(w) or (w.upper() if w else None)
        line2 = f"FL {fl_val}".upper() if fl_val else "FL"
        s_clean = (s[:m.start()] + " " + s[m.end():]).strip()
        s_clean = re.sub(r"\s{2,}", " ", s_clean).strip(" ,")
        return s_clean, line2
    return line1, None

# Also sweep other secondary tokens out of line 1 if they slipped in (SUITE, STE, APT, RM, PMB, etc.)
SEC_TOKENS = ("APARTMENT|APT|SUITE|STE|UNIT|ROOM|RM|DEPARTMENT|DEPT|BUILDING|BLDG|OFFICE|OFC|"
              "PENTHOUSE|PH|LOBBY|LBBY|PMB")
SEC_RE = re.compile(rf"(?i)\b({SEC_TOKENS})\b[ .#-]*([A-Z0-9-]+)?")

def sweep_secondaries_from_line1(line1: str):
    """Remove secondary tokens from line1; return (clean_line1, list of 'ABBR id')."""
    s = " " + (line1 or "") + " "
    items = []
    pos = 0
    while True:
        m = SEC_RE.search(s, pos)
        if not m: break
        typ_raw = m.group(1).upper()
        ident = (m.group(2) or "").upper().strip()
        abbr = SEC_ABBR.get(typ_raw, typ_raw)
        items.append((abbr, ident))
        s = (s[:m.start()] + " " + s[m.end():])  # remove from line1
        pos = 0  # restart, as string changed
    s = re.sub(r"\s{2,}", " ", s).strip(" ,")
    # unique-preserve order
    seen = set(); uniq = []
    for abbr, ident in items:
        key = (abbr, ident)
        if key not in seen:
            seen.add(key); uniq.append((abbr, ident))
    return s, [f"{a} {i}".strip() for a,i in uniq if a or i]

# --- Attention (Pub 28 §214): only from Recipient ---
def attention_from_recipient(recipient: str):
    s = nz(recipient).upper()
    if not s: return None
    m = re.search(r"\b(C/O|CO|CARE OF|ATTN|ATTENTION)\b[ .:]*([A-Z0-9 .,'-]+)", s)
    if not m: return None
    kind, name = m.group(1), re.sub(r"\s+", " ", m.group(2).strip())
    return f"C/O {name}" if kind in {"C/O","CO","CARE OF"} else f"ATTN {name}"

# --- Build helpers from parsed columns (post-usaddress) ---

def build_city_state_zip(row):
    return nz(row.PlaceName).upper(), nz(row.StateName).upper(), normalize_zip(row.ZipCode)

def canonical_secondary(type_str: str, ident: str):
    """Map type to USPS C2 abbreviation + ident (if present)."""
    t = SEC_ABBR.get(nz(type_str).upper(), nz(type_str).upper())
    i = nz(ident).upper()
    if not t and not i:
        return None
    if t == "FL" and i:               # ensure 'FL 08' etc → 'FL 8'
        i = re.sub(r"^0+","", i) or i
    return f"{t} {i}".strip()

def collect_line2_from_columns(row):
    parts = []
    # Occupancy (e.g., SUITE/STE, APT, UNIT, ROOM/RM, PMB, etc.)
    occ = canonical_secondary(getattr(row, "OccupancyType", None), getattr(row, "OccupancyIdentifier", None))
    if occ: parts.append(occ)
    # Subaddress (e.g., FLOOR/FL, BUILDING/BLDG, DEPT/DEPT)
    sub = canonical_secondary(getattr(row, "SubaddressType", None), getattr(row, "SubaddressIdentifier", None))
    if sub: parts.append(sub)
    # unique-preserve order
    seen=set(); out=[]
    for p in parts:
        if p and p not in seen:
            seen.add(p); out.append(p)
    return out

def build_street_line1_from_cols(row):
    """IMPORTANT: Only primary street fields. Do NOT include occupancy/subaddress here."""
    parts = [
        nz(getattr(row, "AddressNumber", None)).upper(), nz(getattr(row, "AddressNumberSuffix", None)).upper(),
        nz(getattr(row, "StreetNamePreDirectional", None)).upper(), nz(getattr(row, "StreetNamePreType", None)).upper(),
        nz(getattr(row, "StreetName", None)).upper(), nz(getattr(row, "StreetNamePostType", None)).upper(),
        nz(getattr(row, "StreetNamePostDirectional", None)).upper(),
    ]
    s = " ".join(p for p in parts if p).strip(" ,")
    return re.sub(r"\s{2,}", " ", s)

def normalize_row(row):
    city, state, zipc = build_city_state_zip(row)
    attn = attention_from_recipient(getattr(row, "Recipient", None))

    # PO Box
    if nz(getattr(row, "USPSBoxType", "")).upper() == "PO BOX" and nz(getattr(row, "USPSBoxID", "")):
        return {
            "address_type": "PO_BOX",
            "address_line_1": f"PO BOX {nz(row.USPSBoxID).upper()}",
            "address_line_2": " ".join(collect_line2_from_columns(row)) or None,
            "city": city, "state": state, "postal_code": zipc, "attention": attn
        }

    # Rural Route (RR <route> BOX <box>)
    if nz(getattr(row, "USPSBoxGroupType", "")).upper() == "RR":
        return {
            "address_type": "RR",
            "address_line_1": f"RR {nz(row.USPSBoxGroupID).upper()} BOX {nz(row.USPSBoxID).upper()}",
            "address_line_2": " ".join(collect_line2_from_columns(row)) or None,
            "city": city, "state": state, "postal_code": zipc, "attention": attn
        }

    # Highway Contract (HC <route> BOX <box>)
    if (nz(getattr(row, "USPSBoxGroupType", "")).upper() == "HC"
        or "HIGHWAY" in nz(getattr(row, "USPSBoxGroupType", "")).upper()
        or "STAR" in nz(getattr(row, "USPSBoxGroupType", "")).upper()):
        return {
            "address_type": "HC",
            "address_line_1": f"HC {nz(row.USPSBoxGroupID).upper()} BOX {nz(row.USPSBoxID).upper()}",
            "address_line_2": " ".join(collect_line2_from_columns(row)) or None,
            "city": city, "state": state, "postal_code": zipc, "attention": attn
        }

    # Street: primary-only line1
    line1 = build_street_line1_from_cols(row)

    # Sweep: other secondaries that accidentally landed in line1 (SUITE/STE/APT/RM/PMB, etc.)
    line1, swept_secs = sweep_secondaries_from_line1(line1)

    # Floors: from text
    line1, fl_from_text = extract_floor_text(line1)

    # Combine secondaries from columns + swept + floor text (dedupe, preserve order)
    line2_parts = collect_line2_from_columns(row)
    if fl_from_text: line2_parts.append(fl_from_text)
    line2_parts.extend(swept_secs)

    # unique-preserve order
    seen=set(); final_line2=[]
    for p in line2_parts:
        p = p.strip()
        if p and p not in seen:
            seen.add(p); final_line2.append(p)

    return {
        "address_type": "STREET",
        "address_line_1": line1,
        "address_line_2": " ".join(final_line2) or None,
        "city": city, "state": state, "postal_code": zipc, "attention": attn
    }

def normalize_parsed_addresses(df: pd.DataFrame, head: int | None = None) -> pd.DataFrame:
    expected = [
        "USPSBoxType","USPSBoxID","USPSBoxGroupType","USPSBoxGroupID",
        "PlaceName","StateName","ZipCode",
        "AddressNumber","StreetName","StreetNamePostType",
        "StreetNamePreDirectional","StreetNamePreType","StreetNamePostDirectional",
        "OccupancyType","OccupancyIdentifier","SubaddressType","SubaddressIdentifier",
        "AddressNumberSuffix","Recipient"
    ]
    for c in expected:
        if c not in df.columns:
            df[c] = None

    work = df if head is None else df.iloc[:head].copy()

    rows = []
    for idx, row in work.iterrows(): # Changed from itertuples(index=True) to iterrows()
        rec = normalize_row(row)
        # Uppercase final textual fields
        for k in ("address_line_1","address_line_2","city","state","attention"):
            if rec.get(k):
                rec[k] = rec[k].upper()
        rec["source_index"] = int(idx)
        rows.append(rec)

    out = pd.DataFrame(
        rows,
        columns=["source_index","address_type","address_line_1","address_line_2",
                 "city","state","postal_code","attention"]
    )
    return out

In [36]:
out_df = normalize_parsed_addresses(parsed_df)
display(out_df.head())

,source_index,address_type,address_line_1,address_line_2,city,state,postal_code,attention
0,0,PO_BOX,PO BOX 92,None,HOSFORD,FL,32334-0092,None
1,1,PO_BOX,PO BOX 296,None,LABELLE,FL,33975-0296,None
2,2,PO_BOX,PO BOX 1860,None,CONROE,TX,77305-1860,None
3,3,STREET,313 5TH ST,None,HUNTINGTN BCH,CA,92648-5119,None
4,4,STREET,3005 S TUTTLE AVE,None,SARASOTA,FL,34239-5511,None


In [37]:
# 1) Ensure no secondary tokens remain in line 1
leaks = out_df[out_df["address_line_1"].str.contains(r"\b(APT|SUITE|STE|UNIT|RM|ROOM|DEPT|BLDG|OFC|PH|LBBY|PMB|FL|FLOOR|FLR)\b", na=False, case=True)]
print("Potential leaks still in line 1:", len(leaks))
display(leaks.head(10))

# 2) Floors were recognized
fl_rows = out_df[out_df["address_line_2"].str.contains(r"\bFL\b", na=False)]
print("Rows with FL in line 2:", len(fl_rows))
display(fl_rows.head(10))

# 3) Suites/Rooms centralized into line 2
sr_rows = out_df[out_df["address_line_2"].str.contains(r"\b(STE|APT|UNIT|RM|PMB)\b", na=False)]
print("Rows with STE/APT/UNIT/RM/PMB in line 2:", len(sr_rows))
display(sr_rows.head(10))

Potential leaks still in line 1: 0


/tmp/ipython-input-577931566.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  leaks = out_df[out_df["address_line_1"].str.contains(r"\b(APT|SUITE|STE|UNIT|RM|ROOM|DEPT|BLDG|OFC|PH|LBBY|PMB|FL|FLOOR|FLR)\b", na=False, case=True)]


,source_index,address_type,address_line_1,address_line_2,city,state,postal_code,attention


Rows with FL in line 2: 1


,source_index,address_type,address_line_1,address_line_2,city,state,postal_code,attention
60,60,STREET,50 CENTRAL AVENUE,FL EIGHTH,SARASOTA,FL,34236-5746,None


Rows with STE/APT/UNIT/RM/PMB in line 2: 33


/tmp/ipython-input-577931566.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sr_rows = out_df[out_df["address_line_2"].str.contains(r"\b(STE|APT|UNIT|RM|PMB)\b", na=False)]


,source_index,address_type,address_line_1,address_line_2,city,state,postal_code,attention
14,14,STREET,21031 VENTURA BLVD,STE 1105,WOODLAND HLS,CA,91364-2256,None
17,17,STREET,14055 RIVEREDGE DR,STE 525,TAMPA,FL,33637-2008,None
19,19,STREET,560 NORTH NIMITZ HIGHWAY,STE 117E,HONOLULU,HI,96817-5330,None
33,33,STREET,45-270 WILLIAM HENRY RD,STE 202,KANEOHE,HI,96744-5808,None
40,40,STREET,900 FORT ST MALL,STE 600,HONOLULU,HI,96813-3701,None
41,41,STREET,45-270 WILLIAM HENRY RD,STE 202,KANEOHE,HI,96744-5808,None
46,46,STREET,74923 US HIGHWAY 111,PMB 170,INDIAN WELLS,CA,92210-7134,None
47,47,STREET,3553 ATLANTIC AVENUE,PMB 350,LONG BEACH,CA,90807-5606,None
48,48,STREET,8325 BROADWAY ST,STE 202 PMB 66,PEARLAND,TX,77581-5773,None
50,50,STREET,6350 LAKE OCONEE PKWY,PMB 133,GREENSBORO,GA,30642-6433,None


In [38]:
# Export to Excel
out_df.to_excel("normalized_addresses.xlsx", index=False)
